In [16]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
from fastapi import FastAPI, Form
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from fastapi.requests import Request
import pandas as pd
pd.options.mode.chained_assignment = None
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report
from collections import defaultdict
import seaborn as sns
import matplotlib.pyplot as plt
import time
from uuid import uuid4
from fastapi.staticfiles import StaticFiles
from preprocessing import data_load
from model_training import model_training
import re

In [17]:
def sequence_mining(team, opponent, df):
    global events_idx
    combined_df = df
    combined_df = combined_df.replace({str(team): 'same', str(opponent): 'other'}, regex=True)

    df = pd.DataFrame()
    encoders = []

    transformed_data = {}
    for column in combined_df.columns[:-1]:
        le = LabelEncoder()
        encoders.append(le)
        transformed_data[column] = le.fit_transform(combined_df[column])

    transformed_df = pd.DataFrame(transformed_data)

    df = pd.concat([transformed_df, combined_df[combined_df.columns[-1]]], axis=1)


    undersample_len = len(df[df['class'] == 1])

    undersample_df = df[df['class'] == 0].sample(n=undersample_len, random_state=43)
    df = pd.concat([df[df['class'] == 1], undersample_df])

    events_idx = {}

    sequence_mining_html = ""

    # Total number of runs (sequences)
    total_sequences = len(combined_df[combined_df['class'] == 1])

    # Prepare lists to store pattern lengths and their corresponding max counts (frequencies)
    pattern_lengths = []
    frequencies = []
    freqs = []

    for j, event in zip(range(12, 112, 11), range(10, 0, -1)):

        event_dict = {}

        a = combined_df.iloc[:, -j:-1][combined_df['class'] == 1]

        # Count occurrences of each row
        row_counts = defaultdict(int)
        for i in range(len(a)):
            row_tuple = tuple(a.iloc[i])
            row_counts[row_tuple] += 1

        # Find the rows with the maximum and second maximum counts
        sorted_row_counts = sorted(row_counts.items(), key=lambda x: x[1], reverse=True)
        mc_row, max_count = sorted_row_counts[0]
        sc_row, second_max_count = sorted_row_counts[1] if len(sorted_row_counts) > 1 else (None, 0)

        # Find all indices of the rows that match the row with the maximum count
        mc_indices = a.apply(lambda row: tuple(row) == mc_row, axis=1)
        mc_indices = mc_indices[mc_indices].index.tolist()

        # Calculate the ratios of the max count and second max count to total sequences
        max_count_ratio = max_count / total_sequences
        second_max_count_ratio = second_max_count / total_sequences if second_max_count > 0 else 0

        events_idx[event] = mc_indices

        # Store the pattern length (event) and frequency (max_count) for ideal length calculation
        pattern_lengths.append(event)
        frequencies.append(max_count)

        # Add sequence mining results to HTML
        sequence_mining_html += f"<p><strong>Last {abs(event-11)} events before run</strong></p>"
        sequence_mining_html += f"<p>Max Count: {max_count}</p>"
        sequence_mining_html += f"<p>Ratio of Max Count to Total Sequences: {max_count_ratio:.2%}</p>"
        sequence_mining_html += f"<p>Second Max Count: {second_max_count}</p>"
        sequence_mining_html += f"<p>Ratio of Second Max Count to Total Sequences: {second_max_count_ratio:.2%}</p>"
        sequence_mining_html += f"<table class='table table-striped'>{combined_df.iloc[mc_indices[0], -j:-1].to_frame().dropna().T.to_html()}</table>"   

        event_dict['Event'] = abs(event-11)
        event_dict['Frequency'] = max_count
        event_dict['Ratio'] = np.round(max_count_ratio, 4)
        event_dict['Sec Frequency'] = second_max_count
        event_dict['Sec Ratio'] = np.round(second_max_count_ratio, 4)
        event_dict['Sequence'] = combined_df.iloc[mc_indices[0], -j:-1].to_frame().dropna().T.to_json()

        freqs.append(event_dict)

    m = pd.DataFrame(freqs) 

    # Calculate the ideal pattern length using the scoring function
    dataset_size = total_sequences

    return m

In [18]:
def analyze_team(team, season, venue):
    # Load and preprocess data
    og_df = pd.read_csv('data/'+season)

    def team_selection(pref_team, df):
        if pref_team in df.HomeTeam.unique():
            pref_df = df[df.HomeTeam == pref_team]
            return pref_df
        else:
            return None
        

    new_df = team_selection(team, og_df)


    # preprocessing code
    factors = ['ShotDist','TimeoutTeam','Substitution', 'Shooter',
               'Rebounder', 'Blocker','Fouler',
               'ReboundType','ViolationPlayer',
               'FreeThrowShooter','TurnoverPlayer']

    fact_cols = [col + str((i // 11) % 10 + 1) for i, col in enumerate(factors * 10)]
    fact_cols.append('class')

    new_df['ShotDist'] = new_df.ShotDist.apply(lambda x: 'close' if x <= 10 else '3pt' if x >= 22 else 'mid' if pd.notna(x) else x)
    
    new_df['TimeoutTeam'] = new_df.apply(
        lambda row: 'timeout_home' if pd.notna(row['TimeoutTeam']) and row['TimeoutTeam'] == row['HomeTeam'] 
        else 'timeout_away' if pd.notna(row['TimeoutTeam']) 
        else row['TimeoutTeam'], 
        axis=1
    )

    new_df['Shooter'] = new_df.apply(lambda row: 'shooter_home' if pd.notna(row['Shooter']) and pd.notna(row['HomePlay'])
                                         else 'shooter_away' if pd.notna(row['Shooter']) and pd.notna(row['AwayPlay'])
                                         else np.nan,
                                         axis=1)

    new_df['Rebounder'] = new_df.apply(lambda row: 'rebounder_home' if pd.notna(row['Rebounder']) and pd.notna(row['HomePlay'])
                                         else 'rebounder_away' if pd.notna(row['Rebounder']) and pd.notna(row['AwayPlay'])
                                         else np.nan,
                                         axis=1)

    new_df['Blocker'] = new_df.apply(lambda row: 'blocker_home' if pd.notna(row['Blocker']) and pd.notna(row['HomePlay'])
                                         else 'blocker_away' if pd.notna(row['Blocker']) and pd.notna(row['AwayPlay'])
                                         else np.nan,
                                         axis=1)

    new_df['Fouler'] = new_df.apply(lambda row: 'fouler_home' if pd.notna(row['Fouler']) and pd.notna(row['HomePlay'])
                                         else 'fouler_away' if pd.notna(row['Fouler']) and pd.notna(row['AwayPlay'])
                                         else np.nan,
                                         axis=1)

    new_df['ViolationPlayer'] = new_df.apply(lambda row: 'violator_home' if pd.notna(row['ViolationPlayer']) and pd.notna(row['HomePlay'])
                                         else 'violator_away' if pd.notna(row['ViolationPlayer']) and pd.notna(row['AwayPlay'])
                                         else np.nan,
                                         axis=1)

    new_df['FreeThrowShooter'] = new_df.apply(lambda row: 'ft_home' if pd.notna(row['FreeThrowShooter']) and pd.notna(row['HomePlay'])
                                         else 'ft_away' if pd.notna(row['FreeThrowShooter']) and pd.notna(row['AwayPlay'])
                                         else np.nan,
                                         axis=1)

    new_df['TurnoverPlayer'] = new_df.apply(lambda row: 'to_player_home' if pd.notna(row['TurnoverPlayer']) and pd.notna(row['HomePlay'])
                                         else 'to_player_away' if pd.notna(row['TurnoverPlayer']) and pd.notna(row['AwayPlay'])
                                         else np.nan,
                                         axis=1)

    new_df['Substitution'] = new_df.apply(lambda row: 'sub_home' if pd.notna(row['EnterGame']) and pd.notna(row['HomePlay'])
                                                  else 'sub_away' if pd.notna(row['EnterGame']) and pd.notna(row['AwayPlay'])
                                                  else np.nan,
                                                  axis=1)

    def home_runner(data):
        global home_runs
        run = []
        home_runs = []
        for idx in data.index:
            if data.at[idx,'HomePlay'] is not np.nan:
                    if 'makes' in data.at[idx,'HomePlay']:
                        run.append(idx)
            elif data.at[idx,'AwayPlay'] is not np.nan:
                    if 'makes' in data.at[idx,'AwayPlay']:
                        run.clear()
            if len(run) == 4:
                home_runs.append(run.copy())
                run.clear()
        return home_runs
                
    home_runner(new_df)

    def away_runner(data):
        global away_runs
        run = []
        away_runs = []
        for idx in data.index:
            if data.at[idx,'AwayPlay'] is not np.nan:
                    if 'makes' in data.at[idx,'AwayPlay']:
                        run.append(idx)
            elif data.at[idx,'HomePlay'] is not np.nan:
                    if 'makes' in data.at[idx,'HomePlay']:
                        run.clear()
            if len(run) == 4:
                away_runs.append(run.copy())
                run.clear()
        return away_runs

    away_runner(new_df)

    all_runs = []
    all_runs.extend(home_runs)
    all_runs.extend(away_runs)

    new_df = new_df[factors]

    def runs_iter(data, runs):
        global runs_df
        runs_df = pd.DataFrame()
        for run in runs:
            a = data.loc[run[0]-10:run[0]-1, factors].values.ravel()
            a = np.append(a,1)
            runs_df = pd.concat([runs_df,pd.DataFrame([a.copy()])])
        return runs_df

    runs_iter(new_df, home_runs)
    runs_df.columns = fact_cols
    runs_df['class'] = runs_df['class'].fillna(1)


    def no_runs_preprocessing(data, runs):
        global no_runs_split

        # find the first index of a run
        r = [i[0] for i in runs]  

        # create a list of runs
        r_x = []
        for num in r:
            r_x.extend(range(num - 10, num + 1))

        # mask the df without runs
        no_runs_df = data[~data.index.isin(r_x)].reset_index(drop=True)

        # segment the df and keep those that are length of 10
        segment_size = 10
        segments = len(no_runs_df) // segment_size

        no_runs_split = np.array_split(no_runs_df, segments)

        no_runs_split = [x for x in no_runs_split if len(x) != 11]

        return no_runs_split

    def no_runs_optimized(data, factors, fact_cols):
        global no_runs_df
        no_runs_df = pd.DataFrame([np.append(segment.loc[:, factors].values.ravel(), int(0)) for segment in data])
        no_runs_df.columns = fact_cols
        return no_runs_df

    no_runs_optimized(no_runs_preprocessing(new_df, home_runs), factors, fact_cols)

    combined_df = pd.concat([runs_df,no_runs_df],ignore_index=True)
    combined_df.to_csv('team_runs/'+str(team)+'_runs.csv', index=True)
    combined_df = pd.read_csv('team_runs/'+str(team)+'_runs.csv',index_col=0)


    if venue == 'home':
        df_data = sequence_mining('home', 'away',combined_df)
    elif venue == 'away':
         df_data = sequence_mining('away', 'home' ,combined_df)

    # df_data = df_data.to_dict(orient="records")

    return df_data


In [19]:
teams = ['DET', 'CLE', 'NOP', 'WAS', 'PHI', 'CHI', 'UTA', 'CHO', 'IND',
       'DEN', 'NYK', 'SAS', 'DAL', 'LAC', 'MIN', 'MEM', 'ATL', 'MIA',
       'OKC', 'TOR', 'BRK', 'GSW', 'LAL', 'POR', 'PHO', 'SAC', 'HOU',
       'MIL', 'ORL', 'BOS']

seasons = ['NBA_PBP_2015-16.csv', 
        'NBA_PBP_2016-17.csv',
        'NBA_PBP_2017-18.csv',
        'NBA_PBP_2018-19.csv',
        'NBA_PBP_2019-20.csv',
        ]

venues = ['home', 'away']

dfs = []

for team in teams:
    for season in seasons:
        for venue in venues:
            try:
                team_mining = analyze_team(team, season, venue)
                team_mining['team'] = team
                team_mining['season'] = re.sub(r'[^0-9-]+','',season)
                team_mining['venue'] = venue
                dfs.append(team_mining)
            except AttributeError:
                print(team, season)

all_data = pd.concat(dfs)


In [20]:
import sqlite3

def initialize_db():
    conn = sqlite3.connect('frequencies.db')
    cursor = conn.cursor()
    cursor.execute('''
                    CREATE TABLE IF NOT EXISTS freqs(
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                event TEXT,
                frequency INTEGER,
                ratio REAL,
                sec_frequency INTEGER,
                sec_ratio REAL,
                team TEXT,
                season TEXT,
                venue TEXT,
                sequence TEXT
                )
                '''
                )
    conn.commit()
    conn.close()

initialize_db()

In [21]:
all_data_dict = all_data.to_dict(orient='records')

records_to_insert = [
    (
        record.get('Event'),
        record.get('Frequency'),
        record.get('Ratio'),
        record.get('Sec Frequency'),
        record.get('Sec Ratio'),
        record.get('team'),
        record.get('season'),
        record.get('venue'),
        record.get('Sequence')
    )
    for record in all_data_dict
]

In [22]:
conn = sqlite3.connect('frequencies.db')
cursor = conn.cursor()

cursor.executemany('''
                    INSERT INTO freqs (
                   event, frequency, ratio, sec_frequency, sec_ratio, team, season, venue, sequence)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                   ''', records_to_insert)

conn.commit()
conn.close()